코드 rf-importance

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import numpy as np

# 학습
rf = RandomForestClassifier(
    n_estimators=500,
    max_features='sqrt',
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print(f"OOB 점수: {rf.oob_score_:.3f}")

# MDI 변수 중요도 (학습 시 자동 계산)
importances_mdi = rf.feature_importances_

# 순열 중요도 (별도 계산)
result = permutation_importance(
    rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)
importances_perm = result.importances_mean

# 상위 20개 변수 비교
top_idx = np.argsort(importances_perm)[-20:]
plt.barh(range(20), importances_perm[top_idx])
plt.yticks(range(20), [feature_names[i] for i in top_idx])
plt.xlabel('순열 중요도'); plt.tight_layout(); plt.show()

코드 xgboost-train

In [ ]:
import xgboost as xgb
from sklearn.metrics import f1_score

# DMatrix 형식 (XGBoost의 효율적 데이터 구조)
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

params = {
    'objective': 'multi:softprob',
    'num_class': 4,
    'learning_rate': 0.05,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,        # L1 정규화
    'reg_lambda': 1.0,        # L2 정규화
    'eval_metric': 'mlogloss'
}

# 조기 종료: 검증 손실이 50라운드 동안 개선되지 않으면 중단
model = xgb.train(
    params, dtrain,
    num_boost_round=2000,
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=50,
    verbose_eval=100
)
print(f"최적 라운드: {model.best_iteration}")

코드 comment-preprocessing

In [ ]:
import re
from konlpy.tag import Mecab
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

mecab = Mecab()

def preprocess_comment(text):
    text = re.sub(r'http\S+', '', text)            # URL 제거
    text = re.sub(r'[ㅋㅎ]{3,}', 'ㅋㅋ', text)        # 자음 반복 정규화
    text = re.sub(r'[ㅠㅜ]{2,}', 'ㅠㅠ', text)
    return text

def korean_tokenizer(text):
    text = preprocess_comment(text)
    pos_tags = mecab.pos(text)
    return [w for w, t in pos_tags
            if t.startswith(('NN', 'VV', 'VA', 'MAG', 'IC'))]

# 1) TF-IDF
vectorizer = TfidfVectorizer(
    tokenizer=korean_tokenizer,
    max_features=5000, min_df=10, ngram_range=(1, 2)
)
X_text = vectorizer.fit_transform(comments)

# 2) 메타 특징
import numpy as np
X_meta = np.array([
    [len(c), c.count('!'), c.count('?'), c.count('ㅋ'), c.count('ㅠ')]
    for c in comments
])

# 3) 결합
X = hstack([X_text, X_meta]).tocsr()
print(f"입력 차원: {X.shape}")

코드 three-models-compare

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score
import lightgbm as lgb
import time

results = {}

# 1) 랜덤포레스트
start = time.time()
rf = RandomForestClassifier(
    n_estimators=500, max_features='sqrt',
    n_jobs=-1, random_state=42
)
rf.fit(X_train, y_train)
results['RF'] = {
    'time': time.time() - start,
    'f1': f1_score(y_test, rf.predict(X_test), average='macro')
}

# 2) LightGBM
start = time.time()
lgb_train = lgb.Dataset(X_train, y_train)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)
params = {
    'objective': 'multiclass', 'num_class': 3,
    'learning_rate': 0.05, 'num_leaves': 63,
    'metric': 'multi_logloss', 'verbose': -1
}
lgbm = lgb.train(params, lgb_train, num_boost_round=2000,
                  valid_sets=[lgb_val],
                  callbacks=[lgb.early_stopping(50)])
y_pred_lgb = lgbm.predict(X_test).argmax(axis=1)
results['LightGBM'] = {
    'time': time.time() - start,
    'f1': f1_score(y_test, y_pred_lgb, average='macro')
}

# 3) 선형 SVM
start = time.time()
svm = LinearSVC(C=1.0, max_iter=3000)
svm.fit(X_train, y_train)
results['LinearSVM'] = {
    'time': time.time() - start,
    'f1': f1_score(y_test, svm.predict(X_test), average='macro')
}

# 결과 출력
for name, r in results.items():
    print(f"{name}: F1={r['f1']:.3f}, 학습시간={r['time']:.1f}초")